# Train JJ 🧠

This notebook turns an open-source base model into **your own JJ model** using LoRA fine-tuning.

**Before you start:** in Colab, go to **Runtime → Change runtime type → T4 GPU** (free).

Steps:
1. Install tools
2. Load JJ's training data
3. Fine-tune the base model (about 5–15 minutes on a free T4)
4. Chat with JJ to test it
5. Save the finished model (to Hugging Face and/or as a GGUF file for the Raspberry Pi)

## 1. Install tools

In [ ]:
!pip install -q "transformers==4.46.3" "trl==0.12.2" "peft==0.13.2" "datasets==3.1.0" "accelerate==1.1.1"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU!")

## 2. Settings

`BASE_MODEL` is the open-source model JJ is built on. Bigger = smarter but slower:
- `Qwen/Qwen2.5-0.5B-Instruct`: tiny, fastest on a Pi
- `Qwen/Qwen2.5-1.5B-Instruct`: good balance (default)
- `Qwen/Qwen2.5-3B-Instruct`: smarter; needs a Pi with 8 GB RAM

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_NAME = "jj-1.5b"
SYSTEM = "You are JJ, a helpful, friendly AI assistant. Answer clearly and concisely."

EPOCHS = 3            # passes over the data; raise to 5 if JJ hasn't learned its identity
LEARNING_RATE = 2e-4
LORA_RANK = 16        # size of the trainable add-on; 8-64 is typical

## 3. Load the training data

Downloads `jj_dataset.jsonl` from your GitHub repo. If the repo is private, instead click the 📁 folder icon on the left and upload `jj_dataset.jsonl`, then skip the `wget` line.

In [ ]:
!wget -q -O jj_dataset.jsonl https://raw.githubusercontent.com/Joseph1-H/test/main/training/jj_dataset.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="jj_dataset.jsonl", split="train")
print(len(dataset), "conversations")
print(dataset[0]["messages"])

## 4. Fine-tune

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    gradient_checkpointing=True,
    max_seq_length=1024,
    report_to="none",
    model_init_kwargs={"torch_dtype": torch.float32},
)

trainer = SFTTrainer(
    model=BASE_MODEL,
    args=args,
    train_dataset=dataset,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

## 5. Chat with your JJ

In [ ]:
tokenizer = trainer.processing_class
model = trainer.model
model.eval()
model.config.use_cache = True

def ask(question, max_new_tokens=200):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

for q in ["Who are you?", "Are you ChatGPT?", "Explain what a GPU is.", "Give me a tip for learning to code."]:
    print("You:", q)
    print("JJ: ", ask(q), "\n")

## 6. Save the finished model

This merges the LoRA add-on into the base model, giving you one standalone JJ model.

In [ ]:
merged = trainer.model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained(OUTPUT_NAME, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_NAME)
print("Saved to", OUTPUT_NAME)

### 6a. Upload to Hugging Face (optional, recommended)

Create a **write** token at huggingface.co → Settings → Access Tokens. Your model is uploaded as a private repo, and the JJ server can then load it by setting `JJ_MODEL_ID=your-username/jj-1.5b`.

In [ ]:
from huggingface_hub import login
login()  # paste your write token when asked

HF_USERNAME = "your-username"   # <- change this
merged.push_to_hub(f"{HF_USERNAME}/{OUTPUT_NAME}", private=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{OUTPUT_NAME}", private=True)
print(f"Uploaded: https://huggingface.co/{HF_USERNAME}/{OUTPUT_NAME}")

### 6b. Convert to GGUF for the Raspberry Pi (optional)

GGUF is the format `llama.cpp` uses. It runs much faster on a Pi than the Python server. `q8_0` keeps nearly full quality at about half the size.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py {OUTPUT_NAME} --outfile {OUTPUT_NAME}-q8_0.gguf --outtype q8_0

from google.colab import files
files.download(f"{OUTPUT_NAME}-q8_0.gguf")

## Next steps
- Add more conversations to `make_dataset.py` (aim for a few hundred good ones) and train again.
- If JJ forgets general knowledge, lower `EPOCHS` or add more general examples.
- If JJ doesn't know its name, raise `EPOCHS` or add more identity examples.